<a href="https://colab.research.google.com/github/cuiandrew08-lab/LiDARFusionLearning/blob/main/CenterPointHead.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import numpy as np

from google.colab import drive
#drive.mount("/content/drive")

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms

import tensorflow as tf

TORCH_version = torch.__version__.split('+')[0]
CUDA_version = torch.version.cuda.replace('.', '')

!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-{TORCH_version}+cu{CUDA_version}.html

import torch_sparse

from scipy.ndimage import maximum_filter

Looking in links: https://data.pyg.org/whl/torch-2.11.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 123.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 92.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 136.0 MB/s eta 0:00:00


In [2]:
def generate_heatmap(w,h, centers,sigma):

  x_axis = torch.arange(0, w, device=centers.device, dtype=torch.float32)
  y_axis = torch.arange(0, h, device=centers.device, dtype=torch.float32)
  y_grid, x_grid = torch.meshgrid(y_axis, x_axis, indexing='ij')

  distances_squared = torch.zeros((w,h), device=centers.device)

  for i in range(centers.shape[0]):
    distances_squared += (x_grid-centers[i][0])**2 +(y_grid-centers[i][1])

  heatmap = torch.exp(-1*distances_squared/(2*sigma**2))

  return heatmap

In [23]:
class CenterPoint_FirstStage(nn.Module): #similar classes(num_classes) are grouped together into one task head. The classes should have similar size, z, yaw, etc. so one task head can be used

  def __init__(self, w, h, C, num_classes, K, hidden_channels=64):
    super().__init__()

    self.w = w
    self.h = h
    self.num_classes = num_classes
    self.C = C
    self.K = K

    self.shared = nn.Sequential(nn.Conv2d(self.C, hidden_channels, kernel_size = 3, padding = 1, bias = True), nn.ReLU)

    self.heat_conv = nn.Conv2d(hidden_channels, num_classes, kernel_size = 1)

    self.offset  = nn.Conv2d(hidden_channels, 2, kernel_size = 1)             # (δx, δy)
    self.z       = nn.Conv2d(hidden_channels, 1, kernel_size = 1)             # z center
    self.size    = nn.Conv2d(hidden_channels, 3, kernel_size = 1)             # log(w, l, h)
    self.yaw     = nn.Conv2d(hidden_channels, 2, kernel_size = 1)             # (sin θ, cos θ)


  def learn_heatmap(self, feature_map):

    heatmap = self.shared(feature_map)
    heatmap = F.sigmoid(self.heat_conv(heatmap))

    return heatmap

  def get_centers(self, heatmap):
    centers_list = []

    for k in range(self.num_classes):
      heatmap_k = heatmap[:,:, k]
      K_count = 0

      local_max_filter = maximum_filter(heatmap_k, size=3, mode='constant', cval=-np.inf)

      is_local_max = (heatmap_k == local_max_filter)

      rows, cols = np.where(is_local_max)

      for i in range(rows.size):
        if K_count < self.K:

          center_0 = torch.tensor([rows[i],cols[i],k])
          centers_list.append(center_0)

          K_count += 1

    centers = torch.stack(centers_list, dim = 0)


  def regression_heads(self, feature_map):

    shared_feats = self.shared(feature_map)

    offset = self.offset(shared_feats)
    z = self.z(shared_feats)
    size = self.size(shared_feats)
    yaw = self.yaw(shared_feats)

    return offset, z, size, yaw

  def forward(self, feature_map):

    heatmap = self.learn_heatmap(feature_map)

    offset, z, size, yaw = self.regression_heads(feature_map)

    centers = get_centers(heatmap)

    for i in range(centers.shape[0]):
      out_list = []

      cx = centers[i][0]
      cy = centers[i][1]
      k = centers[i][2]

      offset_0 = offset[cx][cy]
      z_0 = z[cx][cy]
      size_0 = size[cx][cy]
      yaw_0 = yaw[cx][cy]

      out_0 = [offset_0, z_0, size_0, yaw_0, k, cx, cy] #test first
      out_0 = torch.cat(out_0)

      out_list.append(out_0)

    out = torch.stack(out_list, dim = 0)

    return out

